In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

In [8]:
for root, dirs,files in os.walk('results\\test'):
    for file in files:
        if file.endswith('.csv.csv'):
            oldpath = os.path.join(root, file)
            newpath = os.path.join(root, file.replace('.csv.csv', '.csv'))
            os.rename(oldpath, newpath)

In [52]:
df = pd.read_csv('results\\test\\RidgewithClose\\data_daily_^DJI.csv')
df = df.reset_index()[['actual', 'predict']]

# correct = 0
# for i in range(1, len(df)):
#     correct += (df.actual.iloc[i]<df.actual.iloc[i-1]) == (df.predict.iloc[i]<df.predict.iloc[i-1])
    
# correct/len(df)

def direc_accuracy(y_actual, y_predict):
    correct = 0
    for i in range(1, len(y_actual)):
        correct += (y_predict[i]<y_predict[i-1]) == (y_actual[i]<y_actual[i-1])
        
    return correct/len(y_actual)

print(direc_accuracy(df.actual, df.predict))

0.5074626865671642


In [63]:
path = "results\\test"

results_df = pd.DataFrame(columns=['model','data', 'ticker', 'timeframe', 'datasize', 'MAPE', 'DirecAccuracy', 'MSE'])
for model in os.listdir(path):
    files = os.listdir(os.path.join(path, model))
    for file in files:
        if not file.endswith('.csv'): continue
        
        data = ''
        if file.startswith('data'):
            data, timeframe, ticker = 'data', file.split('_')[1], file.split('_')[2][:-4]
        elif file.startswith('gbm_nl'):
            data, timeframe, ticker = 'gbm_nl', file.split('_')[2], file.split('_')[3][:-4]
        elif file.startswith('gbm'):
            data, timeframe, ticker = 'gbm', file.split('_')[1], file.split('_')[2][:-4]
        elif file.startswith('jump_diffusion'):
            data, timeframe, ticker = 'jump_diffusion', file.split('_')[2], file.split('_')[3][:-4]
            
        df = pd.read_csv(os.path.join(path, model, file))
        
        size = len(df)
        MAPE = mean_absolute_percentage_error(df.actual, df.predict)
        DirecAccuracy = direc_accuracy(df.actual, df.predict)
        MSE = mean_squared_error(df.actual, df.predict)
        
        results_df.loc[len(results_df)] = [model, data, ticker, timeframe, size, MAPE, DirecAccuracy, MSE]

results_df

,model,data,ticker,timeframe,datasize,MAPE,DirecAccuracy,MSE
0,ElasticNetTunedwithClose,data,AAPL,daily,67,0.019409,0.507463,31.919313
1,ElasticNetTunedwithClose,data,BIRET.BO,daily,68,0.006353,0.382353,6.929782
2,ElasticNetTunedwithClose,data,DLF.NS,daily,68,0.017832,0.455882,313.745846
3,ElasticNetTunedwithClose,data,EMBASSY.BO,daily,68,0.008332,0.426471,15.037819
4,ElasticNetTunedwithClose,data,GC=F,daily,69,0.013891,0.347826,3354.139057
...,...,...,...,...,...,...,...,...
3595,XGBwithIndicators,jump_diffusion,OBEROIRLTY.NS,hourly,56,0.006980,0.571429,185.386196
3596,XGBwithIndicators,jump_diffusion,^DJI,hourly,54,0.002319,0.611111,11415.490740
3597,XGBwithIndicators,jump_diffusion,^GSPC,hourly,54,0.007824,0.425926,4824.424424
3598,XGBwithIndicators,jump_diffusion,^N225,hourly,55,0.005715,0.509091,34190.858664


In [67]:
results_df.describe()

,datasize,MAPE,DirecAccuracy,MSE
count,3600.000000,3600.000000,3600.000000,3.600000e+03
mean,87.111111,0.048462,0.493288,6.964343e+05
std,58.159116,0.257429,0.059174,8.809626e+06
min,54.000000,0.001056,0.267857,1.697262e-05
25%,56.000000,0.006246,0.455882,8.861430e+00
50%,67.000000,0.014477,0.494681,2.238474e+02
75%,68.000000,0.032761,0.531915,6.565098e+03
max,315.000000,9.895334,0.678571,2.659644e+08


In [89]:
wmape_df = pd.DataFrame(columns=['model', 'data', 'gbm', 'gbm_nl', 'jump_diffusion'])
for model in results_df.model.unique():
    mapes = []
    for data in ['data', 'gbm', 'gbm_nl', 'jump_diffusion']:
        model_results = results_df[(results_df.model == model) & (results_df.data == data)]
        weighted_MAPE = (model_results.datasize * model_results.MAPE).sum() / model_results.datasize.sum()
        mapes += [weighted_MAPE*100]
    wmape_df.loc[len(wmape_df)] = [model, *mapes]

wmape_df

,model,data,gbm,gbm_nl,jump_diffusion
0,ElasticNetTunedwithClose,0.804677,0.949164,1.212779,0.945303
1,ElasticNetwithClose,18.968168,19.324422,31.228683,14.863536
2,ElasticNetwithIndicators,18.968168,19.324422,31.228683,14.863536
3,KNNTunedwithClose,3.075851,3.261144,2.696319,2.923563
4,KNNwithClose,3.059228,3.153965,2.715098,2.894640
5,KNNwithIndicators,3.059228,3.153965,2.715098,2.894640
6,LGBMTunedwithClose,2.110091,2.224374,1.766412,1.814083
7,LGBMwithClose,2.081230,2.443882,1.789747,1.866944
8,LGBMwithIndicators,2.081230,2.443882,1.789747,1.866944
9,LSTMwithClose,7.786107,8.204340,17.936517,6.826800


In [90]:
direcacc_df = pd.DataFrame(columns=['model', 'data', 'gbm', 'gbm_nl', 'jump_diffusion'])
for model in results_df.model.unique():
    das = []
    for data in ['data', 'gbm', 'gbm_nl', 'jump_diffusion']:
        model_results = results_df[(results_df.model == model) & (results_df.data == data)]
        weighted_DA = (model_results.datasize * model_results.DirecAccuracy).sum() / model_results.datasize.sum()
        das += [weighted_DA]
    direcacc_df.loc[len(direcacc_df)] = [model, *das]

direcacc_df

,model,data,gbm,gbm_nl,jump_diffusion
0,ElasticNetTunedwithClose,0.487245,0.483099,0.500000,0.492666
1,ElasticNetwithClose,0.510523,0.487245,0.493622,0.486607
2,ElasticNetwithIndicators,0.510523,0.487245,0.493622,0.486607
3,KNNTunedwithClose,0.517857,0.482462,0.494260,0.508291
4,KNNwithClose,0.488520,0.478954,0.498087,0.508929
5,KNNwithIndicators,0.488520,0.478954,0.498087,0.508929
6,LGBMTunedwithClose,0.506696,0.500638,0.493941,0.492666
7,LGBMwithClose,0.516901,0.502232,0.494260,0.500638
8,LGBMwithIndicators,0.516901,0.502232,0.494260,0.500638
9,LSTMwithClose,0.492347,0.482781,0.493304,0.487564
